# Demo: End-to-End GNN-BERT Music Context Inference

Loads one audio file, builds its structure graph, runs the fused GNN-BERT model, and prints predicted genre tags.

**Kernel required:** select **Python (gnn-bert-music)** (top-right). Do not use the system Python 3.11 kernel.

In [1]:
import sys, os
from pathlib import Path

# Always resolve paths relative to this notebook's folder
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    # Cursor sometimes starts kernels in repo root
    candidate = NOTEBOOK_DIR / 'notebooks'
    if candidate.exists():
        os.chdir(candidate)
        NOTEBOOK_DIR = candidate

ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(ROOT / 'src'))

print('Python:', sys.executable)
if '.venv' not in sys.executable.replace('\\', '/'):
    raise RuntimeError(
        'Wrong kernel. Select kernel: Python (gnn-bert-music)\n'
        f'Current interpreter: {sys.executable}'
    )

import torch
import yaml

from audio_features import load_audio
from graph_builder import build_segment_similarity_graph
from fusion_model import GNNBertFusionModel

with open(ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Project root:', ROOT)

Python: C:\Users\Mahir\Downloads\gnn-bert-music-context\gnn-bert-music-context\.venv\Scripts\python.exe


Using device: cpu
Project root: C:\Users\Mahir\Downloads\gnn-bert-music-context\gnn-bert-music-context


In [2]:
# --- Step 1: Load audio and build its structure graph ---
AUDIO_PATH = ROOT / 'data' / 'raw' / 'jazz' / 'jazz.00000.wav'
assert AUDIO_PATH.exists(), f'Missing audio file: {AUDIO_PATH}'

y = load_audio(str(AUDIO_PATH), sample_rate=cfg['data']['sample_rate'])
graph = build_segment_similarity_graph(
    y,
    sr=cfg['data']['sample_rate'],
    segment_seconds=cfg['data']['segment_seconds'],
    hop_seconds=cfg['data']['segment_hop_seconds'],
    similarity_threshold=cfg['data']['similarity_threshold'],
    max_similarity_neighbors=cfg['data'].get('max_similarity_neighbors', 3),
)
print(graph)

Data(x=[19, 140], edge_index=[2, 46])


In [3]:
# --- Step 2: Load trained fusion model ---
import json

vocab_path = ROOT / 'data' / 'splits' / 'label_vocab.json'
if vocab_path.exists():
    with open(vocab_path) as f:
        TAG_VOCAB = json.load(f)
else:
    TAG_VOCAB = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']

model = GNNBertFusionModel(
    graph_in_dim=graph.x.shape[1],
    num_labels=len(TAG_VOCAB),
    bert_model_name=cfg['text']['model_name'],
    gnn_hidden_dim=cfg['gnn']['hidden_dim'],
    gnn_out_dim=cfg['gnn']['out_dim'],
    gnn_layers=cfg['gnn']['num_layers'],
    gnn_architecture=cfg['gnn']['architecture'],
    fusion_type=cfg['fusion']['type'],
    fusion_hidden_dim=cfg['fusion']['hidden_dim'],
    bert_finetune=False,
    predict_emotion=True,
).to(device)

ckpt_path = ROOT / 'checkpoints' / 'fusion_model.pt'
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False))
    print('Loaded checkpoint:', ckpt_path)
else:
    print('WARNING: no checkpoint found, using randomly initialized weights (demo only).')

model.eval()
print('num labels:', len(TAG_VOCAB))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint: C:\Users\Mahir\Downloads\gnn-bert-music-context\gnn-bert-music-context\checkpoints\fusion_model.pt
num labels: 10


In [4]:
# --- Step 3: Describe the audio, then run inference ---
# The caption is derived from the waveform (tempo, key, chords, timbre) -- it is
# NOT built from the genre label, so this is a true end-to-end inference path.
from torch_geometric.data import Batch

from preprocess import generate_caption

caption = generate_caption(y, sr=cfg['data']['sample_rate'])

print('Audio-derived caption:')
print(' ', caption, '\n')

graph_batch = Batch.from_data_list([graph]).to(device)
enc = model.bert.tokenize([caption])
input_ids = enc['input_ids'].to(device)
attention_mask = enc['attention_mask'].to(device)

with torch.no_grad():
    out = model(graph_batch.x, graph_batch.edge_index, graph_batch.batch, input_ids, attention_mask)
    probs = torch.sigmoid(out['tag_logits']).cpu().numpy()[0]

print('Predicted tags (probability):')
for tag, p in sorted(zip(TAG_VOCAB, probs), key=lambda x: -x[1]):
    print(f'{tag:15s} {p:.3f}')

true_genre = AUDIO_PATH.stem.split('.')[0]
print(f"\nTrue genre:      {true_genre}")
print(f"Predicted (top): {TAG_VOCAB[int(probs.argmax())]}")

if out['valence'] is not None:
    print(
        "\nValence/arousal heads are present but untrained: this dataset pairing has no "
        "DEAM annotations, so those targets are null and masked out of the loss. "
        "The numbers below are therefore not meaningful."
    )
    print(f"  raw valence head: {out['valence'].item():.2f}")
    print(f"  raw arousal head: {out['arousal'].item():.2f}")

Audio-derived caption:
  A moderately paced recording at about 123 BPM in C major, with a warm, rounded timbre and a dense stream of note onsets. Its harmony centres on C minor, F minor and F major, the texture balances sustained harmony with percussive attacks, and loudness swings strongly across the clip, with a clean, tonal spectrum. 



Predicted tags (probability):
classical       0.607
jazz            0.150
disco           0.042
rock            0.029
reggae          0.027
pop             0.023
country         0.005
blues           0.004
hiphop          0.002
metal           0.000

True genre:      jazz
Predicted (top): classical

Valence/arousal heads are present but untrained: this dataset pairing has no DEAM annotations, so those targets are null and masked out of the loss. The numbers below are therefore not meaningful.
  raw valence head: -0.16
  raw arousal head: 1.09
